# 02d — Làm sạch nội dung bài và build lại 1.000 URL

Chạy trên **CPU** sau khi code đã được merge vào `main`. Notebook đọc raw `stage-a-laodong-recovered-v1` hiện có, dùng quy tắc cắt nội dung phụ mới và ghi build riêng `stage-a-data-v5-clean-text`. Không sửa raw hay build v3. Chạy từ đầu trong runtime Colab mới.


Notebook tùy chọn, không thuộc luồng chính 00–04. Bootstrap dùng optional_data_code_lock.json riêng. Runtime mới chạy lại từ đầu để resume. Khi chạy team, cả ba ghim cùng full SHA vào CODE_REVISION trước khi crawl. Build bằng code đã merge dùng tên run mới; notebook 03 phải dùng cùng full CODE_COMMIT khi freeze.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "optional_data_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "optional_data_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Xác minh raw 1.000 URL đã lưu

In [ ]:
from vietmedbridge.artifacts import read_json
from vietmedbridge.crawl import read_completed_crawl
from vietmedbridge.dataset import load_snapshot, parquet_path

SNAPSHOT = load_snapshot(DATA_ROOT)
PILOT = read_json(DATA_ROOT / 'reports/inventory/stage_a.json')
CRAWL_RUN = 'stage-a-laodong-recovered-v1'
BUILD_RUN = 'stage-a-data-v5-clean-text-reviewed'
OLD_BUILD_RUN = 'stage-a-data-v3-laodong'
RAW = read_completed_crawl(
    DATA_ROOT / 'crawl' / CRAWL_RUN,
    links_path=DATA_ROOT / PILOT['files']['stage_a_links']['path'],
    origin_corpus_sha256=SNAPSHOT['files']['links_corpus.parquet']['sha256'],
)
print('Code commit:', CODE_COMMIT)
print('Raw đã lưu:', RAW['recorded_documents'], '/', RAW['requested_input_records'])
print('Có đủ 1.000 outcome:', RAW['range_complete'])
assert RAW['range_complete'] and RAW['recorded_documents'] == 1000


## 2. Build mới từ raw cũ

In [ ]:
from vietmedbridge.build import build_corpus
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer

TOKENIZER_LOCK = read_json(DATA_ROOT / 'tokenizer_lock.json')
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(TOKENIZER_LOCK['revision'])
PIPELINE_CONFIG = json.loads((CHECKOUT / 'configs/data_pipeline.json').read_text())
BUILD = build_corpus(
    DATA_ROOT / 'crawl' / CRAWL_RUN, DATA_ROOT / 'processed',
    TOKENIZER, TOKENIZER_SPEC, run_name=BUILD_RUN,
    config=ChunkConfig(**PIPELINE_CONFIG['chunking']), work_dir=WORK_DIR,
    official_links=parquet_path(DATA_ROOT, 'links_corpus.parquet'),
)
print('Build:', BUILD_RUN)
print(json.dumps(BUILD['counts'], ensure_ascii=False, indent=2))


## 3. So sánh build mới với v3

Số document có thể giảm nếu trang chủ hoặc mã rác bị loại. Đừng coi số document lớn hơn là bằng chứng chất lượng tốt hơn.


In [ ]:
import pyarrow.dataset as ds
import pandas as pd
from vietmedbridge.health import health_report
from vietmedbridge.validation import artifact_paths
from vietmedbridge.validation import artifact_paths

def document_frame(run_name):
    root = DATA_ROOT / 'processed' / run_name
    paths = artifact_paths(root, read_json(root / 'build.json'), 'documents')
    if not paths:
        raise FileNotFoundError(f'Không tìm thấy documents của {run_name}')
    return ds.dataset([str(path) for path in paths], format='parquet').to_table(
        columns=['doc_id', 'url', 'title', 'language', 'language_method',
                 'quality_tier', 'quality_flags', 'source_text_sha256', 'source_text']
    ).to_pandas()

OLD = document_frame(OLD_BUILD_RUN)
NEW = document_frame(BUILD_RUN)
old_ids, new_ids = set(OLD.doc_id), set(NEW.doc_id)
print('V3 documents:', len(OLD), '| V5 documents:', len(NEW))
print('Không còn trong V5:', len(old_ids - new_ids), sorted(old_ids - new_ids)[:30])
print('Mới trong V5:', len(new_ids - old_ids), sorted(new_ids - old_ids)[:30])
print('Ngôn ngữ V5:')
display(NEW.language.value_counts().rename_axis('language').reset_index(name='documents'))
print('Quality tier V5:')
display(NEW.quality_tier.value_counts().rename_axis('quality_tier').reset_index(name='documents'))
HEALTH = health_report(
    DATA_ROOT / 'processed' / BUILD_RUN,
    official_links=parquet_path(DATA_ROOT, 'links_corpus.parquet'),
    crawl_dir=DATA_ROOT / 'crawl' / CRAWL_RUN,
    work_dir=WORK_DIR, audit_size=PIPELINE_CONFIG['audit_size'],
)
print('Failure reasons:')
display(pd.DataFrame(HEALTH['failure_reasons']))
print('Health report:', DATA_ROOT / 'processed' / BUILD_RUN / 'reports')


## 4. Kiểm tra dấu hiệu nội dung phụ trong V3 và V5

Các số sau là dấu hiệu tự động, không thay cho việc xem lại một số bài theo domain.


In [ ]:
import re
from collections import Counter

MARKERS = {
    'youlai_doctor': r'医生推荐',
    'familydoctor_related': r'相关文章推荐|相关推荐|相关问答',
    'medlatec_service': r'Bình luận \(\)|Lựa chọn dịch vụ',
    'phutho_template': r'\{(?:title|publish|head|name|body)\}',
    'pharmacity_related': r'Các bài viết liên quan',
    'nghean_related': r'\| TIN LIÊN QUAN \|',
    'replacement_char': '�',
}

def marker_counts(frame):
    return {name: int(frame.source_text.str.contains(pattern, flags=re.I, regex=True, na=False).sum())
            for name, pattern in MARKERS.items()}

print('V3 document markers:', marker_counts(OLD))
print('V5 document markers:', marker_counts(NEW))
child_root = DATA_ROOT / 'processed' / BUILD_RUN
child_paths = artifact_paths(child_root, read_json(child_root / 'build.json'), 'children')
children = ds.dataset([str(path) for path in child_paths], format='parquet').to_table(
    columns=['doc_id', 'text']).to_pandas()
print('V5 chunks:', len(children))
print('V5 chunk markers:', {name: int(children.text.str.contains(pattern, flags=re.I, regex=True, na=False).sum())
                           for name, pattern in MARKERS.items()})
print('V5 flags:', Counter(flag for flags in NEW.quality_flags for flag in flags).most_common(15))


## Đọc kết quả

Gửi bảng `V3 documents / V5 documents`, `Ngôn ngữ V5`, `Quality tier V5` và `Failure reasons` để đánh giá. Nếu cần xem từng bài bị loại, dùng `OLD[OLD.doc_id.isin(old_ids - new_ids)]` trong ô mới. Sau đó chạy notebook 03 với `CRAWL_RUN = 'stage-a-laodong-recovered-v1'`, `BUILD_RUN = 'stage-a-data-v5-clean-text-reviewed'`; nếu notebook 03 có ô audit tự thêm từng trỏ cố định vào build v2/v3, sửa đường dẫn trong ô đó sang v5. Chưa chạy 10k chỉ dựa vào số lượng document.

Ô kiểm tra ngay trên đếm các dấu hiệu nội dung phụ trong toàn bộ document và chunk của V3/V5. Chỉ nâng lên 10.000 URL sau khi xem cả số bài giữ được và các dấu hiệu còn sót.
